# Mini Data Mesh: Dos Dominios con Catálogo Compartido

Objetivo: Construir un mini data mesh de ejemplo con dos dominios (Ventas y Marketing), publicando productos de datos descubiertos a través de un catálogo compartido. Mostramos contratos simples, calidad básica, y una capa semántica ligera para interoperabilidad.

---
## 1. Diseño Lógico
- Dominio Ventas → Producto: orders_daily (dataset de pedidos diarios)
- Dominio Marketing → Producto: campaign_performance (métricas de campañas)
- Catálogo Compartido: índice de productos con metadatos, contratos y etiquetas semánticas
- Capa Semántica: glosario mínimo (Customer, Order, Channel, Campaign)

---
## 2. Contratos de Datos (estructura mínima)
Usaremos un formato Python (dict) para simplificar, con campos, tipos, descripciones, SLAs y reglas de calidad.

<!-- course-nav -->
> Ruta: [Inicio del repositorio](../README.md) | [Índice del nivel](README.md)  
> Anterior: [Frameworks de Big Data de Última Generación (2025) + Comparativo de Desempeño](16_frameworks_big_data_modernos_comparativo.ipynb) | Siguiente: [Observabilidad, Metadata y Linaje para Ecosistemas IA/Big Data (Simulación)](18_observabilidad_metadata_linaje_ia_bigdata_simulacion.ipynb)


<!-- course-quality-structure -->
## Prerrequisitos

- Haber revisado los notebooks anteriores de la ruta o contar con experiencia equivalente.
- Conocer los conceptos básicos mencionados en el README del nivel.
- Tener disponible el repositorio local y las dependencias de `requirements.txt` cuando se ejecuten celdas de código.

## Preparación

1. Trabaja desde la raíz del repositorio.
2. Activa tu entorno de Python.
3. Instala dependencias con `pip install -r requirements.txt` si aún no lo hiciste.
4. Ejecuta las celdas en orden para conservar el contexto de variables y resultados.

## Mapa del notebook

1. 1. Diseño Lógico
2. 2. Contratos de Datos (estructura mínima)
3. 3. Interoperabilidad vía Capa Semántica Ligera
4. 4. Checks de Calidad Básicos (como código)
5. 5. Publicación y Descubrimiento
6. 6. Próximos pasos

## Diagrama de continuidad

```mermaid
flowchart LR
    A["Notebook anterior"] --> B["Mini Data Mesh: Dos Dominios con Catálogo Compartido"] --> C["Notebook siguiente"]
```


In [ ]:
from typing import Dict, Any, List
from dataclasses import dataclass, field
from datetime import datetime, timezone
import pandas as pd

@dataclass
class Contract:
    name: str
    schema: List[Dict[str, Any]]
    sla_freshness_minutes: int
    quality_rules: Dict[str, float]
    semantics: Dict[str, str]

@dataclass
class DataProduct:
    domain: str
    name: str
    owner: str
    contract: Contract
    tags: List[str] = field(default_factory=list)
    created_at: str = field(default_factory=lambda: datetime.now(timezone.utc).isoformat())

class Catalog:
    def __init__(self):
        self.products: Dict[str, DataProduct] = {}

    def register(self, dp: DataProduct):
        key = f'{dp.domain}.{dp.name}'
        if key in self.products:
            raise ValueError(f'Data product ya registrado: {key}')
        self.products[key] = dp

    def search(self, text: str) -> List[str]:
        text = text.lower()
        return [key for key, dp in self.products.items()
                if text in key.lower() or any(text in tag.lower() for tag in dp.tags)]

    def by_concept(self, concept: str) -> List[str]:
        return [key for key, dp in self.products.items() if concept in dp.contract.semantics.values()]

    def show(self):
        return pd.DataFrame([
            {'key': key, 'domain': dp.domain, 'name': dp.name, 'owner': dp.owner,
             'freshness_sla_m': dp.contract.sla_freshness_minutes, 'tags': ','.join(dp.tags)}
            for key, dp in self.products.items()
        ])

orders_contract = Contract(
    'orders_daily',
    [{'name': 'order_id', 'type': 'string'}, {'name': 'customer_id', 'type': 'string'},
     {'name': 'order_amount', 'type': 'decimal'}, {'name': 'order_date', 'type': 'date'}],
    30,
    {'completitud': 0.99, 'duplicados_max': 0.001},
    {'customer_id': 'Customer', 'order_amount': 'Revenue'}
)
campaign_contract = Contract(
    'campaign_performance',
    [{'name': 'campaign_id', 'type': 'string'}, {'name': 'spend', 'type': 'decimal'},
     {'name': 'clicks', 'type': 'int'}],
    45,
    {'completitud': 0.98, 'duplicados_max': 0.001},
    {'spend': 'MarketingSpend', 'clicks': 'Engagement'}
)

catalog = Catalog()
catalog.register(DataProduct('sales', 'orders_daily', 'owner_sales', orders_contract, ['sales', 'revenue']))
catalog.register(DataProduct('marketing', 'campaign_performance', 'owner_marketing', campaign_contract, ['marketing', 'campaigns']))
print(catalog.show())
print('Búsqueda por texto:', catalog.search('sales'))
print('Búsqueda por concepto Revenue:', catalog.by_concept('Revenue'))

---
## 3. Interoperabilidad vía Capa Semántica Ligera
Consultamos por concepto (e.g., 'Channel') y unimos datasets por un concepto común.

In [ ]:
# Búsqueda por concepto
catalog.by_concept('Channel')

In [ ]:
# Simulamos datasets físicos (mini dataframes)
import pandas as pd
orders_df = pd.DataFrame([
    {'order_id':'o1','customer_id':'u1','order_amount':120.0,'order_date':'2025-11-01','channel':'online'},
    {'order_id':'o2','customer_id':'u2','order_amount':75.0,'order_date':'2025-11-01','channel':'store'},
    {'order_id':'o3','customer_id':'u1','order_amount':30.0,'order_date':'2025-11-02','channel':'online'}
])
campaign_df = pd.DataFrame([
    {'campaign_id':'c1','channel':'online','impressions':1000,'clicks':50,'spend':200.0},
    {'campaign_id':'c2','channel':'store','impressions':800,'clicks':32,'spend':120.0}
])
# Unión por concepto 'Channel' (mapeado a campo 'channel')
merged = orders_df.merge(campaign_df, on='channel', how='left')
# KPI ejemplo: ROAS (ingresos / spend) por canal
kpi = merged.groupby('channel').agg(revenue=('order_amount','sum'), spend=('spend','sum'))
kpi['ROAS'] = kpi['revenue']/kpi['spend']
kpi.reset_index()

---
## 4. Checks de Calidad Básicos (como código)
Aplicamos reglas del contrato para validar datasets simulados.

In [ ]:
def check_quality(df: pd.DataFrame, rules: Dict[str, float]):
    results = {}
    if 'completitud' in rules:
        completeness = 1 - (df.isna().any(axis=1).sum()/len(df))
        results['completitud'] = completeness
    if 'duplicados_max' in rules:
        dups = df.duplicated().mean()
        results['duplicados_max'] = dups
    return results

q_orders = check_quality(orders_df, orders_contract.quality_rules)
q_campaign = check_quality(campaign_df, campaign_contract.quality_rules)
q_orders, q_campaign

---
## 5. Publicación y Descubrimiento
- Los dominios publican contratos y metadatos en el catálogo.
- Los consumidores encuentran productos por texto, tags o conceptos semánticos.
- La interoperabilidad se logra al acordar conceptos comunes (glosario) y mapearlos.

### Búsqueda por texto

In [ ]:
catalog.search('sales'), catalog.search('campaign'), catalog.search('orders')

---
## 6. Próximos pasos
- Expandir contratos a formatos serializables (YAML/JSON).
- Integrar validación de esquemas (pydantic/great expectations).
- Hacer *policy-as-code* para acceso y privacidad.
- Usar catálogo real (DataHub/Amundsen) y un *registry* de data products.

<!-- course-quality-close -->
## Ejercicio de cierre

Revisa el caso trabajado en el notebook y documenta una decisión de arquitectura: contexto, opción elegida, alternativas descartadas, riesgos y métricas de validación.

## Evaluación

- El notebook se ejecuta en orden sin depender de estado oculto.
- Las decisiones técnicas están justificadas con trade-offs claros.
- Los resultados, diagramas o tablas apoyan la explicación y no reemplazan el razonamiento.
- Las referencias usadas son oficiales, primarias o están incluidas en la bibliografía del curso.

## Referencias

- Referencias del curso: ../04-recursos/referencias.md
